# ML - Groupby - EDA


In [ ]:
# [Raw Master.parquet]
#         |
#         v
#    🔍 EDA
#         |
#         v
#  Timestamp 파싱 + 정렬
#         |
#         v
#  time-based split 기준(ts cut) 계산
#         |
#         v
#  split 컬럼 부여 (train / val / test)
#         |
#         v
#  🔍 EDA (전체 분포 / 라벨 비율 / sanity check)
#         |
#         v
#  bucket_ts 생성 (ts → 1h truncate)
#         |
#         v
#  🔍 EDA (시간/배치 단위 분포 확인)
#         |
#         v
#  row-wise 피처 엔지니어링 (⚠️ 규칙 엄격)
#         |
#         v
#  모델에 안 넣을 컬럼 drop
#         |
#         v
#  마지막에 pandas로 collect
#         |
#         v
#  평가 함수 정의 custom
#         |
#         v
#  W&B Sweeps용 train/eval 함수 정의
#         |
#         v
#  sweep Config 정의 + 실행

# 0) Install & Login & Drive Mount

In [ ]:
!pip install -q wandb polars xgboost imblearn

In [ ]:
import wandb
wandb.login()
# 계정 만들었으면, 2 누르고 본인 api 키 입력

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: hyo9 (0326byeol-korea-ac-kr) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 1) Imports

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    roc_auc_score, roc_curve,
    average_precision_score, precision_recall_curve,  # AUPRC/PR curve 핵심
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score
)

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from collections import Counter
import random

import joblib
import pickle
import os

# 2) 테이블 병합(Polars + Parquet)



In [ ]:
master_parquet = "/content/drive/MyDrive/data/HI-Medium_Master.parquet"

q_master = pl.scan_parquet(master_parquet)

In [ ]:
q_master.head(5).collect()

Timestamp,From Bank,From Account,To Bank,To Account,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,Amount_Received_USD,Amount_Paid_USD,Sender_Entity,Sender_Bank_Name,Receiver_Entity,Receiver_Bank_Name
str,i64,str,i64,str,f64,str,f64,str,str,i64,f64,f64,str,str,str,str
"""2022/09/01 00:09""",1313,"""803E47FD0""",181605,"""8249FE0E0""",96289.46,"""US Dollar""",96289.46,"""US Dollar""","""Cheque""",0,96289.46,96289.46,"""Corporation #6642""","""Savings Bank of Indianapolis""","""Partnership #37115""","""First Bank of Boston"""
"""2022/09/01 00:03""",1313,"""803E47FD0""",181605,"""8249FE0E0""",19780.24,"""US Dollar""",19780.24,"""US Dollar""","""Credit Card""",0,19780.24,19780.24,"""Corporation #6642""","""Savings Bank of Indianapolis""","""Partnership #37115""","""First Bank of Boston"""
"""2022/09/01 00:14""",1313,"""803E47FD0""",181605,"""8249FE0E0""",65287.84,"""US Dollar""",65287.84,"""US Dollar""","""Cash""",0,65287.84,65287.84,"""Corporation #6642""","""Savings Bank of Indianapolis""","""Partnership #37115""","""First Bank of Boston"""
"""2022/09/01 00:01""",1313,"""803E47FD0""",181605,"""8249FE0E0""",6554.4,"""US Dollar""",6554.4,"""US Dollar""","""ACH""",0,6554.4,6554.4,"""Corporation #6642""","""Savings Bank of Indianapolis""","""Partnership #37115""","""First Bank of Boston"""
"""2022/09/01 00:05""",1313,"""803E47FD0""",181605,"""8249FE0E0""",4242.09,"""US Dollar""",4242.09,"""US Dollar""","""Wire""",0,4242.09,4242.09,"""Corporation #6642""","""Savings Bank of Indianapolis""","""Partnership #37115""","""First Bank of Boston"""


In [ ]:
q_master = q_master.with_columns(
    pl.col("Timestamp")
      .str.strptime(pl.Datetime, format="%Y/%m/%d %H:%M", strict=False)
      .alias("ts")
)

In [ ]:
# '문자열'이었던 컬럼을 '날짜형'으로 덮어쓰기
q_master = q_master.with_columns(
    pl.col("Timestamp").str.to_datetime().alias("Timestamp")
)

In [ ]:
# 컬럼명과 타입을 딕셔너리 형태로 반환
print(q_master.schema)

Schema({'Timestamp': Datetime(time_unit='us', time_zone=None), 'From Bank': Int64, 'From Account': String, 'To Bank': Int64, 'To Account': String, 'Amount Received': Float64, 'Receiving Currency': String, 'Amount Paid': Float64, 'Payment Currency': String, 'Payment Format': String, 'Is Laundering': Int64, 'Amount_Received_USD': Float64, 'Amount_Paid_USD': Float64, 'Sender_Entity': String, 'Sender_Bank_Name': String, 'Receiver_Entity': String, 'Receiver_Bank_Name': String, 'ts': Datetime(time_unit='us', time_zone=None)})


##🔍 EDA 1차

In [ ]:
# 1️⃣ 데이터 볼륨 & 기본 구조

import polars as pl

# row 수 / column 수
basic_info = q_master.select([
    pl.count().alias("row_count"),
    pl.lit(len(q_master.columns)).alias("column_count")
]).collect()

basic_info


row_count,column_count
u32,i32
31898669,18


In [ ]:
q_master.schema

Schema([('Timestamp', Datetime(time_unit='us', time_zone=None)),
        ('From Bank', Int64),
        ('From Account', String),
        ('To Bank', Int64),
        ('To Account', String),
        ('Amount Received', Float64),
        ('Receiving Currency', String),
        ('Amount Paid', Float64),
        ('Payment Currency', String),
        ('Payment Format', String),
        ('Is Laundering', Int64),
        ('Amount_Received_USD', Float64),
        ('Amount_Paid_USD', Float64),
        ('Sender_Entity', String),
        ('Sender_Bank_Name', String),
        ('Receiver_Entity', String),
        ('Receiver_Bank_Name', String),
        ('ts', Datetime(time_unit='us', time_zone=None))])

In [ ]:
# 기간 범위 (Timestamp / ts 둘 다 체크)

time_range = q_master.select([
    pl.col("Timestamp").min().alias("min_timestamp"),
    pl.col("Timestamp").max().alias("max_timestamp"),
    pl.col("ts").min().alias("min_ts"),
    pl.col("ts").max().alias("max_ts"),
]).collect()

time_range


min_timestamp,max_timestamp,min_ts,max_ts
datetime[μs],datetime[μs],datetime[μs],datetime[μs]
2022-09-01 00:00:00,2022-09-28 15:58:00,2022-09-01 00:00:00,2022-09-28 15:58:00


In [ ]:
# 중복 row 존재 여부

dup_count = (
    q_master
    .select(pl.count().alias("total"))
    .join(
        q_master.unique().select(pl.count().alias("unique")),
        how="cross"
    )
    .with_columns(
        (pl.col("total") - pl.col("unique")).alias("duplicate_rows")
    )
    .collect()
)

dup_count


total,unique,duplicate_rows
u32,u32,u32
31898669,31898649,20


In [ ]:
# Account / Transaction Unique 수

account_stats = q_master.select([
    pl.col("From Account").n_unique().alias("unique_from_accounts"),
    pl.col("To Account").n_unique().alias("unique_to_accounts"),
]).collect()

account_stats


unique_from_accounts,unique_to_accounts
u32,u32
2013627,1689925


In [ ]:
# 2️⃣ 라벨 분포 (Is Laundering)

label_dist = (
    q_master
    .group_by("Is Laundering")
    .agg(pl.count().alias("tx_count"))
    .with_columns(
        (pl.col("tx_count") / pl.col("tx_count").sum() * 100)
        .alias("ratio_percent")
    )
    .collect()
)

label_dist

Is Laundering,tx_count,ratio_percent
i64,u32,f64
0,31863439,99.889557
1,35230,0.110443


In [ ]:
# Fraud rate 한 줄 요약

fraud_rate = q_master.select([
    pl.mean("Is Laundering").alias("fraud_rate")
]).collect()

fraud_rate

fraud_rate
f64
0.001104


In [ ]:
#3️⃣ 금액 분포 점검
#기본 통계
# 컬럼을 선택해서 데이터를 가져온 후(collect), 전체 요약 통계(describe)를 계산합니다.
amount_stats = (
    q_master
    .select([
        pl.col("Amount Paid"),
        pl.col("Amount Received")
    ])
    .collect()
    .describe()
)

amount_stats

statistic,Amount Paid,Amount Received
str,f64,f64
"""count""",3.1898669e7,3.1898669e7
"""null_count""",0.0,0.0
"""mean""",4.4175e6,6.4310e6
"""std""",1.8483e9,2.5927e9
"""min""",0.000001,0.000001
"""25%""",209.22,207.86
"""50%""",1471.51,1469.23
"""75%""",11757.54,11835.01
"""max""",8.1586e12,8.1586e12


In [ ]:
# 0원 / 음수 거래 체크
amount_anomaly = q_master.select([
    (pl.col("Amount Paid") == 0).sum().alias("paid_zero"),
    (pl.col("Amount Received") == 0).sum().alias("received_zero"),
    (pl.col("Amount Paid") < 0).sum().alias("paid_negative"),
    (pl.col("Amount Received") < 0).sum().alias("received_negative"),
]).collect()

amount_anomaly


paid_zero,received_zero,paid_negative,received_negative
u32,u32,u32,u32
0,0,0,0


In [ ]:
# 상위 1% cutoff

amount_cutoff = q_master.select([
    pl.col("Amount Paid").quantile(0.99).alias("paid_p99"),
    pl.col("Amount Received").quantile(0.99).alias("received_p99"),
]).collect()

amount_cutoff


paid_p99,received_p99
f64,f64
1.0650e7,1.1503e7


In [ ]:
# 4️⃣ 통화 / 포맷 분포
# Payment Currency

currency_dist = (
    q_master
    .group_by("Payment Currency")
    .agg(pl.count().alias("tx_count"))
    .sort("tx_count", descending=True)
    .collect()
)

currency_dist


Payment Currency,tx_count
str,u32
"""US Dollar""",11688326
"""Euro""",7316012
"""Yuan""",2330785
"""Shekel""",1408707
"""Canadian Dollar""",1078358
…,…
"""Mexican Peso""",841283
"""Rupee""",738745
"""Bitcoin""",688914


In [ ]:
#Receiving Currency

recv_currency_dist = (
    q_master
    .group_by("Receiving Currency")
    .agg(pl.count().alias("tx_count"))
    .sort("tx_count", descending=True)
    .collect()
)

recv_currency_dist


Receiving Currency,tx_count
str,u32
"""US Dollar""",11594393
"""Euro""",7329224
"""Yuan""",2295924
"""Shekel""",1428622
"""Canadian Dollar""",1089398
…,…
"""Mexican Peso""",852298
"""Rupee""",741621
"""Bitcoin""",698372


In [ ]:
# Payment Format

format_dist = (
    q_master
    .group_by("Payment Format")
    .agg(pl.count().alias("tx_count"))
    .sort("tx_count", descending=True)
    .collect()
)

format_dist

Payment Format,tx_count
str,u32
"""Cheque""",12280121
"""Credit Card""",8778042
"""ACH""",3868513
"""Cash""",3217537
"""Reinvestment""",1945644
"""Wire""",1119774
"""Bitcoin""",689038


In [ ]:
#5️⃣ Timestamp 원본 품질 체크
# null 존재 여부

null_check = q_master.select([
    pl.col("Timestamp").null_count().alias("timestamp_nulls"),
    pl.col("ts").null_count().alias("ts_nulls"),
]).collect()

null_check


timestamp_nulls,ts_nulls
u32,u32
0,0


In [ ]:
#동일 timestamp 과다 여부 (Top 10)
ts_dup = (
    q_master
    .group_by("Timestamp")
    .agg(pl.count().alias("tx_count"))
    .sort("tx_count", descending=True)
    .head(10)
    .collect()
)

ts_dup


Timestamp,tx_count
datetime[μs],u32
2022-09-01 00:14:00,44215
2022-09-01 00:15:00,44188
2022-09-01 00:20:00,44066
2022-09-01 00:23:00,44009
2022-09-01 00:21:00,43973
2022-09-01 00:24:00,43925
2022-09-01 00:06:00,43903
2022-09-01 00:07:00,43893
2022-09-01 00:10:00,43872


In [ ]:
# 초 단위 존재 여부 확인

second_check = q_master.select([
    pl.col("Timestamp").dt.second().n_unique().alias("unique_seconds")
]).collect()

second_check


unique_seconds
u32
1


# 4) Timestamp

In [ ]:
# =========================================================
# Timestamp 파싱 + 정렬 준비
#   - 원본 Timestamp(string) 유지
#   - ts(Datetime) 새로 생성
#   - ts 파싱 실패(null) 개수 출력
# =========================================================
total_rows = q_master.select(pl.len()).collect().item()

# q_master = q_master.with_columns(
#     pl.col("Timestamp")
#       .str.strptime(pl.Datetime, format="%Y/%m/%d %H:%M", strict=False)
#       .alias("ts")
# )

null_ts_rows = (
    q_master.filter(pl.col("ts").is_null())
     .select(pl.len())
     .collect()
     .item()
)

valid_rows = total_rows - null_ts_rows

print(f"Total rows                    : {total_rows:,}")
print(f"Rows with NULL ts (parse fail): {null_ts_rows:,}")
print(f"Valid ts rows                 : {valid_rows:,}")

# split은 ts가 유효한 행만 대상으로 진행 (추천)
q_valid = q_master.filter(pl.col("ts").is_not_null())

Total rows                    : 31,898,669
Rows with NULL ts (parse fail): 0
Valid ts rows                 : 31,898,669


In [ ]:
# =========================================================
# time-based split 기준(ts cut) 계산 (60:20:20)
#   - collect 최소화: 경계 ts 2개만 뽑음
# =========================================================
n = q_valid.select(pl.len()).collect().item()
train_end_idx = int(n * 0.6)
val_end_idx   = int(n * 0.8)

q_sorted = q_valid.sort("ts")

train_cut_ts = (
    q_sorted.select("ts")
            .slice(train_end_idx, 1)
            .collect()
            .item()
)

val_cut_ts = (
    q_sorted.select("ts")
            .slice(val_end_idx, 1)
            .collect()
            .item()
)

print("Train cutoff ts:", train_cut_ts)
print("Val cutoff ts  :", val_cut_ts)

Train cutoff ts: 2022-09-09 20:43:00
Val cutoff ts  : 2022-09-14 05:46:00


In [ ]:
# =========================================================
# split 컬럼 부여 (train/val/test)
# =========================================================
q_split = q_sorted.with_columns(
    pl.when(pl.col("ts") <= train_cut_ts)
      .then(pl.lit("train"))
      .when(pl.col("ts") <= val_cut_ts)
      .then(pl.lit("val"))
      .otherwise(pl.lit("test"))
      .alias("split")
)

# sanity check: split 분포
split_counts = (
    q_split.group_by("split")
           .agg(pl.len().alias("rows"))
           .collect()
)
print(split_counts)

# 라벨 비율 sanity check (optional)
label_counts = (
    q_split.group_by(["split", "Is Laundering"])
           .agg(pl.len().alias("rows"))
           .collect()
)
print(label_counts)

shape: (3, 2)
┌───────┬──────────┐
│ split ┆ rows     │
│ ---   ┆ ---      │
│ str   ┆ u32      │
╞═══════╪══════════╡
│ val   ┆ 6379980  │
│ test  ┆ 6378958  │
│ train ┆ 19139731 │
└───────┴──────────┘
shape: (6, 3)
┌───────┬───────────────┬──────────┐
│ split ┆ Is Laundering ┆ rows     │
│ ---   ┆ ---           ┆ ---      │
│ str   ┆ i64           ┆ u32      │
╞═══════╪═══════════════╪══════════╡
│ val   ┆ 1             ┆ 9054     │
│ train ┆ 1             ┆ 15536    │
│ test  ┆ 0             ┆ 6368318  │
│ train ┆ 0             ┆ 19124195 │
│ test  ┆ 1             ┆ 10640    │
│ val   ┆ 0             ┆ 6370926  │
└───────┴───────────────┴──────────┘


##🔍 EDA 2차
####(전체 분포 / 라벨 비율 / sanity check)


In [ ]:
# Split 비율 검증 (Row + Fraud)
# 이미 count는 했으니까 비율까지 한 번에

split_ratio = (
    q_split
    .group_by("split")
    .agg([
        pl.len().alias("rows"),
        pl.sum("Is Laundering").alias("fraud_count"),
    ])
    .with_columns([
        (pl.col("rows") / pl.col("rows").sum() * 100).alias("row_ratio_%"),
        (pl.col("fraud_count") / pl.col("rows") * 100).alias("fraud_rate_%"),
    ])
    .sort("split")
    .collect()
)

split_ratio

split,rows,fraud_count,row_ratio_%,fraud_rate_%
str,u32,i64,f64,f64
"""test""",6378958,10640,19.997568,0.166798
"""train""",19139731,15536,60.00166,0.081171
"""val""",6379980,9054,20.000772,0.141913


In [ ]:
# 2️⃣ Split 라벨 분포 Pivot

split_label_pivot = (
    q_split
    .group_by(["split", "Is Laundering"])
    .agg(pl.len().alias("rows"))
    .collect()
    .pivot(
        values="rows",
        index="split",
        columns="Is Laundering"
    )
    .fill_null(0)
)

split_label_pivot


split,1,0
str,u32,u32
"""test""",10640,6368318
"""val""",9054,6370926
"""train""",15536,19124195


In [ ]:
#3️⃣ 시간 순 Fraud Drift
#월별 fraud rate 확인.

monthly_fraud_trend = (
    q_split
    .with_columns(
        pl.col("ts").dt.truncate("1mo").alias("month")
    )
    .group_by("month")
    .agg([
        pl.len().alias("tx_count"),
        pl.mean("Is Laundering").alias("fraud_rate")
    ])
    .sort("month")
    .collect()
)

monthly_fraud_trend

month,tx_count,fraud_rate
datetime[μs],u32,f64
2022-09-01 00:00:00,31898669,0.001104


In [ ]:
# 4️⃣ 금액 분포 Drift (Split 간)
# 평균 / 중앙값 비교.

amount_drift = (
    q_split
    .group_by("split")
    .agg([
        pl.mean("Amount_Paid_USD").alias("paid_mean"),
        pl.median("Amount_Paid_USD").alias("paid_median"),
        pl.mean("Amount_Received_USD").alias("recv_mean"),
        pl.median("Amount_Received_USD").alias("recv_median"),
    ])
    .collect()
)

amount_drift


split,paid_mean,paid_median,recv_mean,recv_median
str,f64,f64,f64,f64
"""train""",381471.386485,868.4874,386154.047324,868.48
"""test""",335941.17849,926.9,335436.433111,926.9265
"""val""",212312.681787,708.3655,213011.919865,708.3153


In [ ]:
# 5️⃣ 신규 계좌 비율 (Inductive 여부)

# Step 1 — Train 계좌 set
train_accounts = (
    q_split
    .filter(pl.col("split") == "train")
    .select(pl.col("From Account"))
    .unique()
)

In [ ]:
# Step 2 — Test 등장 계좌
test_accounts = (
    q_split
    .filter(pl.col("split") == "test")
    .select(pl.col("From Account"))
    .unique()
)

In [ ]:
# Step 3 — 신규 계좌 계산

new_accounts_ratio = (
    test_accounts
    .join(train_accounts, on="From Account", how="anti")
    .select(pl.count().alias("new_accounts"))
    .join(
        test_accounts.select(pl.count().alias("total_test_accounts")),
        how="cross"
    )
    .with_columns(
        (pl.col("new_accounts") / pl.col("total_test_accounts") * 100)
        .alias("new_account_ratio_%")
    )
    .collect()
)

new_accounts_ratio


new_accounts,total_test_accounts,new_account_ratio_%
u32,u32,f64
9427,928618,1.015164


In [ ]:
# 5️⃣-2 신규 계좌 비율 (Inductive 여부) (val포함)

# 1. 공부할 때 본 놈들 (Train + Val)
seen_accounts = (
    q_split
    .filter(pl.col("split").is_in(["train", "val"]))
    .select(pl.col("From Account"))
    .unique()
)

# 2. 실전에서 처음 만난 놈들 (Test)
pure_new_accounts_ratio = (
    test_accounts
    .join(seen_accounts, on="From Account", how="anti")
    .select(pl.count().alias("pure_new_accounts"))
    .join(
        test_accounts.select(pl.count().alias("total_test_accounts")),
        how="cross"
    )
    .with_columns(
        (pl.col("pure_new_accounts") / pl.col("total_test_accounts") * 100)
        .alias("pure_new_account_ratio_%")
    )
    .collect()
)

print("--- [엄격한 기준: Train+Val에도 없던 진짜 신규 계좌 비율] ---")
pure_new_accounts_ratio

--- [엄격한 기준: Train+Val에도 없던 진짜 신규 계좌 비율] ---


pure_new_accounts,total_test_accounts,pure_new_account_ratio_%
u32,u32,f64
7625,928618,0.821113


In [ ]:


bank_drift = (
    q_split
    .group_by(["split", "From Bank"])
    .agg(pl.len().alias("tx_count"))
    .sort(["split", "tx_count"], descending=True)
    .collect()
)

bank_drift

split,From Bank,tx_count
str,i64,u32
"""val""",70,637923
"""val""",12,34973
"""val""",11,34389
"""val""",0,34155
"""val""",20,32946
…,…,…
"""test""",3125097,1
"""test""",3150935,1
"""test""",3218599,1


In [ ]:
# 1. 전체 데이터(q_split)에서 은행별 건수 및 비율 계산
bank_stats = (
    q_split
    .group_by("From Bank")
    .agg(pl.len().alias("count"))
    .with_columns(
        (pl.col("count") / pl.col("count").sum() * 100).round(2).alias("ratio_%")
    )
    .sort("count", descending=True)
    .collect()
)

print("--- [전체 데이터 기준: 은행별 거래 점유율 현황] ---")
print(bank_stats)

# 2. (선택 사항) 만약 사기 거래만 따로 떼어서 은행 비중을 보고 싶다면
fraud_bank_stats = (
    q_split
    .filter(pl.col("Is Laundering") == 1)
    .group_by("From Bank")
    .agg(pl.len().alias("fraud_count"))
    .with_columns(
        (pl.col("fraud_count") / pl.col("fraud_count").sum() * 100).round(2).alias("fraud_ratio_%")
    )
    .sort("fraud_count", descending=True)
    .collect()
)

print("\n--- [사기 거래 데이터 기준: 은행별 점유율 현황] ---")
print(fraud_bank_stats)

--- [전체 데이터 기준: 은행별 거래 점유율 현황] ---
shape: (122_330, 3)
┌───────────┬─────────┬─────────┐
│ From Bank ┆ count   ┆ ratio_% │
│ ---       ┆ ---     ┆ ---     │
│ i64       ┆ u32     ┆ f64     │
╞═══════════╪═════════╪═════════╡
│ 70        ┆ 2960807 ┆ 9.28    │
│ 12        ┆ 166485  ┆ 0.52    │
│ 11        ┆ 164584  ┆ 0.52    │
│ 0         ┆ 159908  ┆ 0.5     │
│ 20        ┆ 157093  ┆ 0.49    │
│ …         ┆ …       ┆ …       │
│ 3141296   ┆ 1       ┆ 0.0     │
│ 3152729   ┆ 1       ┆ 0.0     │
│ 3223874   ┆ 1       ┆ 0.0     │
│ 3205927   ┆ 1       ┆ 0.0     │
│ 3213641   ┆ 1       ┆ 0.0     │
└───────────┴─────────┴─────────┘

--- [사기 거래 데이터 기준: 은행별 점유율 현황] ---
shape: (4_108, 3)
┌───────────┬─────────────┬───────────────┐
│ From Bank ┆ fraud_count ┆ fraud_ratio_% │
│ ---       ┆ ---         ┆ ---           │
│ i64       ┆ u32         ┆ f64           │
╞═══════════╪═════════════╪═══════════════╡
│ 70        ┆ 4190        ┆ 11.89         │
│ 20        ┆ 300         ┆ 0.85          │
│ 0  

In [ ]:
bank_name_drift = (
    q_split
    .group_by(["split", "Sender_Bank_Name"]) # 2번째 줄에 바로 적용
    .agg(pl.len().alias("count"))
    .with_columns(
        (pl.col("count") / pl.col("count").sum().over("split") * 100).round(2).alias("ratio_%")
    )
    .collect()
)

# 보기 편하게 피벗 테이블로 변환
bank_name_pivot = (
    bank_name_drift
    .pivot(
        values="ratio_%",
        index="Sender_Bank_Name",
        on="split"
    )
    .fill_null(0)
    .sort("train", descending=True)
)

print("--- [은행 이름별 시점 점유율 변화 (Drift 체크)] ---")
print(bank_name_pivot)

--- [은행 이름별 시점 점유율 변화 (Drift 체크)] ---
shape: (80_179, 4)
┌───────────────────────────────┬───────┬───────┬───────┐
│ Sender_Bank_Name              ┆ train ┆ test  ┆ val   │
│ ---                           ┆ ---   ┆ ---   ┆ ---   │
│ str                           ┆ f64   ┆ f64   ┆ f64   │
╞═══════════════════════════════╪═══════╪═══════╪═══════╡
│ National Bank of Indianapolis ┆ 8.95  ┆ 10.04 ┆ 10.12 │
│ Bank of Miami                 ┆ 0.53  ┆ 0.57  ┆ 0.56  │
│ National Bank of Los Angeles  ┆ 0.51  ┆ 0.54  ┆ 0.54  │
│ France Bank #62               ┆ 0.51  ┆ 0.54  ┆ 0.55  │
│ Hearthstone Bancorp           ┆ 0.49  ┆ 0.53  ┆ 0.54  │
│ …                             ┆ …     ┆ …     ┆ …     │
│ Belgium Bank #3777            ┆ 0.0   ┆ 0.0   ┆ 0.0   │
│ UK Bank #188                  ┆ 0.0   ┆ 0.0   ┆ 0.0   │
│ Greece Bank #2507             ┆ 0.0   ┆ 0.0   ┆ 0.0   │
│ China Bank #1740              ┆ 0.0   ┆ 0.0   ┆ 0.0   │
│ India Bank #828               ┆ 0.0   ┆ 0.0   ┆ 0.0   │
└──────────────

In [ ]:
# 1. 시점(split), 레이블(Is Laundering), 은행명 기준 집계 및 비중 계산
bank_full_drift = (
    q_split
    .group_by(["split", "Is Laundering", "Sender_Bank_Name"])
    .agg(pl.len().alias("count"))
    .with_columns(
        # 각 split 내에서 레이블(0, 1)별로 은행 비중 계산
        (pl.col("count") / pl.col("count").sum().over(["split", "Is Laundering"]) * 100)
        .round(2)
        .alias("ratio_%")
    )
    .collect()
)

# 2. 보기 편하게 피벗: 행은 은행, 열은 (Split + Label) 조합
# 멀티 인덱스처럼 보기 위해 컬럼명을 깔끔하게 가공합니다.
bank_full_pivot = (
    bank_full_drift
    .with_columns(
        # 'train_정상', 'test_사기' 식의 컬럼명을 만들기 위한 문자열 결합
        (pl.col("split") + "_" + pl.col("Is Laundering").cast(pl.String))
        .alias("column_name")
    )
    .pivot(
        values="ratio_%",
        index="Sender_Bank_Name",
        on="column_name"
    )
    .fill_null(0)
    .sort("train_0", descending=True) # 정상 거래(train_0) 비중 순으로 정렬
)

print("--- [정상(0) vs 사기(1) 시점별 은행 점유율 통합 비교] ---")
# 보기 좋게 주요 컬럼 순서 조정 (Train -> Val -> Test 순)
columns_order = ["Sender_Bank_Name", "train_0", "train_1", "val_0", "val_1", "test_0", "test_1"]
print(bank_full_pivot.select([c for c in columns_order if c in bank_full_pivot.columns]))

--- [정상(0) vs 사기(1) 시점별 은행 점유율 통합 비교] ---
shape: (80_179, 7)
┌───────────────────────────────┬─────────┬─────────┬───────┬───────┬────────┬────────┐
│ Sender_Bank_Name              ┆ train_0 ┆ train_1 ┆ val_0 ┆ val_1 ┆ test_0 ┆ test_1 │
│ ---                           ┆ ---     ┆ ---     ┆ ---   ┆ ---   ┆ ---    ┆ ---    │
│ str                           ┆ f64     ┆ f64     ┆ f64   ┆ f64   ┆ f64    ┆ f64    │
╞═══════════════════════════════╪═════════╪═════════╪═══════╪═══════╪════════╪════════╡
│ National Bank of Indianapolis ┆ 8.95    ┆ 15.56   ┆ 10.12 ┆ 10.05 ┆ 10.05  ┆ 8.75   │
│ Bank of Miami                 ┆ 0.53    ┆ 0.63    ┆ 0.56  ┆ 0.76  ┆ 0.57   ┆ 0.91   │
│ National Bank of Los Angeles  ┆ 0.51    ┆ 0.89    ┆ 0.54  ┆ 0.96  ┆ 0.54   ┆ 0.72   │
│ France Bank #62               ┆ 0.51    ┆ 0.52    ┆ 0.55  ┆ 0.61  ┆ 0.54   ┆ 0.66   │
│ Hearthstone Bancorp           ┆ 0.49    ┆ 0.78    ┆ 0.54  ┆ 0.63  ┆ 0.53   ┆ 0.99   │
│ …                             ┆ …       ┆ …       ┆ …    

# 6) bucket_ts 생성 (1h)

In [ ]:
# =========================================================
# bucket_ts 생성 (ts → 1h truncate)
#   - baseline v1: 거래(row) 단위 모델링이지만,
#     이후 확장을 위해 bucket_ts는 만들어 둠
# =========================================================
q_feat = q_split.with_columns(
    pl.col("ts").dt.truncate("1h").alias("bucket_ts")
).with_columns(
    pl.col("bucket_ts").dt.strftime("%Y-%m-%d %H:%M:%S").alias("bucket_ts_str")
)

# bucket 분포 sanity check (optional)
bucket_check = (
    q_feat.filter(pl.col("split") == "train")
          .select([
              pl.col("bucket_ts").min().alias("train_bucket_min"),
              pl.col("bucket_ts").max().alias("train_bucket_max"),
              pl.len().alias("train_rows"),
          ])
          .collect()
)
print(bucket_check)

shape: (1, 3)
┌─────────────────────┬─────────────────────┬────────────┐
│ train_bucket_min    ┆ train_bucket_max    ┆ train_rows │
│ ---                 ┆ ---                 ┆ ---        │
│ datetime[μs]        ┆ datetime[μs]        ┆ u32        │
╞═════════════════════╪═════════════════════╪════════════╡
│ 2022-09-01 00:00:00 ┆ 2022-09-09 20:00:00 ┆ 19139731   │
└─────────────────────┴─────────────────────┴────────────┘


In [ ]:
#“노드 키” 만들기 (송신/수신 각각)
q_feat = q_feat.with_columns([
    (pl.col("From Account").cast(pl.Utf8) + "_" + pl.col("bucket_ts_str").cast(pl.Utf8)).alias("sender_node"),
    (pl.col("To Account").cast(pl.Utf8)   + "_" + pl.col("bucket_ts_str").cast(pl.Utf8)).alias("receiver_node"),
])

##🔍 EDA 3차
#### (시간/배치 단위 분포 확인)

In [ ]:
# 1️⃣ 시간대별 거래량 분포
# bucket별 거래 수

bucket_volume = (
    q_feat
    .group_by("bucket_ts")
    .agg(pl.len().alias("tx_count"))
    .sort("bucket_ts")
    .collect()
)

bucket_volume


bucket_ts,tx_count
datetime[μs],u32
2022-09-01 00:00:00,1380972
2022-09-01 01:00:00,133998
2022-09-01 02:00:00,133882
2022-09-01 03:00:00,133467
2022-09-01 04:00:00,134226
…,…
2022-09-27 18:00:00,2
2022-09-27 20:00:00,1
2022-09-28 11:00:00,3


In [ ]:
# 1️⃣ 시간대별 거래량 분포
# bucket별 거래 수

bucket_volume = (
    q_feat
    .group_by("bucket_ts")
    .agg(pl.len().alias("tx_count"))
    .sort("tx_count", descending=True)
    .collect()
)

bucket_volume.head(10)


bucket_ts,tx_count
datetime[μs],u32
2022-09-01 00:00:00,1380972
2022-09-16 00:00:00,183323
2022-09-02 00:00:00,182572
2022-09-09 00:00:00,166505
2022-09-07 00:00:00,137681
2022-09-06 00:00:00,137561
2022-09-14 00:00:00,136941
2022-09-08 00:00:00,136909
2022-09-12 00:00:00,136808


In [ ]:
# 1️⃣ 시간대별 거래량 분포
# bucket별 거래 수

bucket_volume = (
    q_feat
    .group_by("bucket_ts")
    .agg(pl.len().alias("tx_count"))
    .sort("tx_count", descending=True)
    .collect()
)

bucket_volume.head(20)


bucket_ts,tx_count
datetime[μs],u32
2022-09-01 00:00:00,1380972
2022-09-16 00:00:00,183323
2022-09-02 00:00:00,182572
2022-09-09 00:00:00,166505
2022-09-07 00:00:00,137681
…,…
2022-09-01 09:00:00,134668
2022-09-01 11:00:00,134662
2022-09-01 12:00:00,134493


In [ ]:
# 사기 거래만 따로 떼서 사기거래 건수 별로 보기
fraud_bucket_volume = (
    q_feat
    .filter(pl.col("Is Laundering") == 1)
    .group_by("bucket_ts")
    .agg(pl.len().alias("fraud_tx_count"))
    .sort("fraud_tx_count", descending=True)
    .collect()
)

fraud_bucket_volume.head(20)

bucket_ts,fraud_tx_count
datetime[μs],u32
2022-09-13 15:00:00,164
2022-09-14 12:00:00,161
2022-09-15 11:00:00,158
2022-09-16 13:00:00,155
2022-09-08 12:00:00,152
…,…
2022-09-14 11:00:00,142
2022-09-10 13:00:00,141
2022-09-13 13:00:00,141


insight: 00:00에 거래가 많은거로 봐서는 자동거래일것같음      
사기거래 자동이체가 32퍼였음        
허나 사기거래는 주로 낮에 이루어짐    

In [ ]:
# 분포 통계

bucket_volume_stats = (
    q_feat
    .group_by("bucket_ts")
    .agg(pl.len().alias("tx_count"))
    .select([
        pl.mean("tx_count").alias("mean_tx"),
        pl.median("tx_count").alias("median_tx"),
        pl.max("tx_count").alias("max_tx"),
        pl.min("tx_count").alias("min_tx"),
    ])
    .collect()
)

bucket_volume_stats


mean_tx,median_tx,max_tx,min_tx
f64,f64,u32,u32
51284.033762,32415.0,1380972,1


In [ ]:
# 2️⃣ bucket별 Fraud 분포
# Fraud count per bucket

bucket_fraud = (
    q_feat
    .group_by("bucket_ts")
    .agg([
        pl.len().alias("tx_count"),
        pl.sum("Is Laundering").alias("fraud_count"),
        pl.mean("Is Laundering").alias("fraud_rate"),
    ])
    .sort("bucket_ts")
    .collect()
)

bucket_fraud


bucket_ts,tx_count,fraud_count,fraud_rate
datetime[μs],u32,i64,f64
2022-09-01 00:00:00,1380972,68,0.000049
2022-09-01 01:00:00,133998,43,0.000321
2022-09-01 02:00:00,133882,42,0.000314
2022-09-01 03:00:00,133467,29,0.000217
2022-09-01 04:00:00,134226,43,0.00032
…,…,…,…
2022-09-27 18:00:00,2,1,0.5
2022-09-27 20:00:00,1,1,1.0
2022-09-28 11:00:00,3,1,0.333333


In [ ]:
# 3️⃣ bucket 크기 안정성

# 그래프 학습 가능성 판단 핵심.

bucket_size_dist = (
    q_feat
    .group_by("bucket_ts")
    .agg(pl.len().alias("tx_count"))
    .select([
        pl.col("tx_count").quantile(0.01).alias("p01"),
        pl.col("tx_count").quantile(0.25).alias("p25"),
        pl.col("tx_count").median().alias("p50"),
        pl.col("tx_count").quantile(0.75).alias("p75"),
        pl.col("tx_count").quantile(0.99).alias("p99"),
    ])
    .collect()
)

bucket_size_dist


p01,p25,p50,p75,p99
f64,f64,f64,f64,f64
1.0,28.0,32415.0,78321.0,136941.0


In [ ]:
# 4️⃣ Active Account per bucket

# 그래프 노드 크기 추정.

active_accounts = (
    q_feat
    .group_by("bucket_ts")
    .agg([
        pl.col("From Account").n_unique().alias("active_senders"),
        pl.col("To Account").n_unique().alias("active_receivers"),
    ])
    .with_columns(
        (pl.col("active_senders") + pl.col("active_receivers"))
        .alias("total_active_accounts")
    )
    .sort("bucket_ts")
    .collect()
)

active_accounts

bucket_ts,active_senders,active_receivers,total_active_accounts
datetime[μs],u32,u32,u32
2022-09-01 00:00:00,1002863,972410,1975273
2022-09-01 01:00:00,80851,85566,166417
2022-09-01 02:00:00,80772,85458,166230
2022-09-01 03:00:00,80522,85074,165596
2022-09-01 04:00:00,80793,85547,166340
…,…,…,…
2022-09-27 18:00:00,1,2,3
2022-09-27 20:00:00,1,1,2
2022-09-28 11:00:00,1,2,3


In [ ]:
active_accounts = (
    q_feat
    .group_by("bucket_ts")
    .agg([
        pl.col("From Account").n_unique().alias("active_senders"),
        pl.col("To Account").n_unique().alias("active_receivers"),
        # 송신자와 수신자를 합친 뒤 유니크한 값만 추출
        pl.concat_list(["From Account", "To Account"])
          .flatten()
          .n_unique()
          .alias("total_unique_accounts")
    ])
    .sort("bucket_ts")
    .collect()
)

In [ ]:
active_accounts

bucket_ts,active_senders,active_receivers,total_unique_accounts
datetime[μs],u32,u32,u32
2022-09-01 00:00:00,1002863,972410,1092269
2022-09-01 01:00:00,80851,85566,120683
2022-09-01 02:00:00,80772,85458,120271
2022-09-01 03:00:00,80522,85074,119679
2022-09-01 04:00:00,80793,85547,120270
…,…,…,…
2022-09-27 18:00:00,1,2,2
2022-09-27 20:00:00,1,1,2
2022-09-28 11:00:00,1,2,2


결과: 약 **88만 명(45%)**의 계정이 해당 기간 내에 돈을 보내기도 하고 받기도 했다는 뜻

In [ ]:
#4-2 total_unique_accounts로 정렬

active_accounts_sorted = (
    q_feat
    .group_by("bucket_ts")
    .agg([
        pl.col("From Account").n_unique().alias("active_senders"),
        pl.col("To Account").n_unique().alias("active_receivers"),
        pl.concat_list(["From Account", "To Account"])
          .flatten()
          .n_unique()
          .alias("total_unique_accounts")
    ])
    .sort("total_unique_accounts", descending=True) # 높은 순 정렬
    .collect()
)

active_accounts_sorted.head()

bucket_ts,active_senders,active_receivers,total_unique_accounts
datetime[μs],u32,u32,u32
2022-09-01 00:00:00,1002863,972410,1092269
2022-09-16 00:00:00,110764,112312,216776
2022-09-02 00:00:00,110813,112489,216608
2022-09-09 00:00:00,105648,105458,206059
2022-09-07 00:00:00,88061,86639,170709


In [ ]:
# 5️⃣ 신규 계좌 등장 (Temporal Inductive)

account_first_seen = (
    q_feat
    .group_by("From Account")
    .agg(pl.col("bucket_ts").min().alias("first_seen"))
)

new_accounts_per_bucket = (
    account_first_seen
    .group_by("first_seen")
    .agg(pl.len().alias("new_accounts"))
    .sort("first_seen")
    .collect()
)

new_accounts_per_bucket

first_seen,new_accounts
datetime[μs],u32
2022-09-01 00:00:00,1002863
2022-09-01 01:00:00,45154
2022-09-01 02:00:00,43109
2022-09-01 03:00:00,40882
2022-09-01 04:00:00,39006
…,…
2022-09-21 00:00:00,1
2022-09-21 08:00:00,2
2022-09-21 14:00:00,1


In [ ]:
# 6️⃣ Scatter / Gather 전조 탐지

# (간단 fan-out / fan-in)

# Fan-out (동시간 다수 송금)

fan_out = (
    q_feat
    .group_by(["bucket_ts", "From Account"])
    .agg(pl.len().alias("out_degree"))
    .filter(pl.col("out_degree") >= 5)
    .sort("out_degree", descending=True)
    .collect()
)

fan_out


bucket_ts,From Account,out_degree
datetime[μs],str,u32
2022-09-01 00:00:00,"""100428660""",10909
2022-09-01 00:00:00,"""1004286A8""",6911
2022-09-02 11:00:00,"""100428660""",4617
2022-09-16 15:00:00,"""100428660""",4549
2022-09-16 07:00:00,"""100428660""",4438
…,…,…
2022-09-16 07:00:00,"""81032C920""",5
2022-09-02 04:00:00,"""80D6F7CC0""",5
2022-09-09 14:00:00,"""80046D370""",5


In [ ]:
fan_out.head(10)

bucket_ts,From Account,out_degree
datetime[μs],str,u32
2022-09-01 00:00:00,"""100428660""",10909
2022-09-01 00:00:00,"""1004286A8""",6911
2022-09-02 11:00:00,"""100428660""",4617
2022-09-16 15:00:00,"""100428660""",4549
2022-09-16 07:00:00,"""100428660""",4438
2022-09-02 14:00:00,"""100428660""",4422
2022-09-16 06:00:00,"""100428660""",4411
2022-09-02 03:00:00,"""100428660""",4408
2022-09-02 00:00:00,"""100428660""",4404


In [ ]:
# Fan-in (동시간 다수 수신)

fan_in = (
    q_feat
    .group_by(["bucket_ts", "To Account"])
    .agg(pl.len().alias("in_degree"))
    .filter(pl.col("in_degree") >= 5)
    .sort("in_degree", descending=True)
    .collect()
)

fan_in


bucket_ts,To Account,in_degree
datetime[μs],str,u32
2022-09-09 03:00:00,"""100428660""",119
2022-09-02 13:00:00,"""100428660""",115
2022-09-02 01:00:00,"""100428660""",113
2022-09-09 13:00:00,"""100428660""",110
2022-09-02 04:00:00,"""100428660""",110
…,…,…
2022-09-14 02:00:00,"""8365DC650""",5
2022-09-01 04:00:00,"""82935DAD0""",5
2022-09-08 14:00:00,"""8103210A0""",5


In [ ]:
fan_in.head(10)

bucket_ts,To Account,in_degree
datetime[μs],str,u32
2022-09-09 03:00:00,"""100428660""",119
2022-09-02 13:00:00,"""100428660""",115
2022-09-02 01:00:00,"""100428660""",113
2022-09-09 13:00:00,"""100428660""",110
2022-09-02 04:00:00,"""100428660""",110
2022-09-09 19:00:00,"""100428660""",109
2022-09-09 11:00:00,"""100428660""",107
2022-09-09 23:00:00,"""100428660""",106
2022-09-16 12:00:00,"""100428660""",105


In [ ]:
# Amount_Paid_USD가 높은 기준 시간대가 언제일까
# 거래량 상위 10개 bucket_ts 추출
top_10_buckets = (
    q_feat
    .group_by("bucket_ts")
    .agg(pl.len().alias("tx_count"))
    .sort("tx_count", descending=True)
    .head(10)
    .collect()
    .get_column("bucket_ts")
    .to_list()
)

# 2. 해당 버킷들에 속한 상세 거래 내역 추출
top_bucket_details = (
    q_feat
    .filter(pl.col("bucket_ts").is_in(top_10_buckets))
    .select([
        pl.col("Timestamp"),               # 정확한 초 단위 시간
        pl.col("Sender_Bank_Name"),        # 송신 은행
        pl.col("Receiver_Bank_Name"),      # 수신 은행
        pl.col("Amount_Paid_USD"),         # USD 금액
        pl.col("From Account"),            # 보내는 놈
        pl.col("To Account"),              # 받는 놈
        pl.col("Payment Currency"),        # 통화
        pl.col("Is Laundering")            # 사기 여부 (실제 정답지)
    ])
    .sort(["Amount_Paid_USD"], descending=True)      # 시간순 정렬
    .collect()
)

# 결과 출력 (너무 많을 수 있으니 상위 10개만 먼저 확인)
print(f"--- [거래량 Top 10 버킷 내 상세 내역] ---")
top_bucket_details.head(10)

--- [거래량 Top 10 버킷 내 상세 내역] ---


Timestamp,Sender_Bank_Name,Receiver_Bank_Name,Amount_Paid_USD,From Account,To Account,Payment Currency,Is Laundering
datetime[μs],str,str,f32,str,str,str,i64
2022-09-01 00:11:00,"""Portugal Bank #100""","""Russia Bank #90""",1.3870e11,"""8046EAFD0""","""81D650470""","""Ruble""",0
2022-09-01 00:11:00,"""Portugal Bank #100""","""Portugal Bank #100""",8.8594e10,"""8046EAFD0""","""8046EAFD0""","""Euro""",0
2022-09-01 00:15:00,"""Portugal Bank #100""","""Russia Bank #90""",7.9733e10,"""8046EAFD0""","""81D650470""","""Ruble""",0
2022-09-01 00:15:00,"""Portugal Bank #100""","""Portugal Bank #100""",5.0930e10,"""8046EAFD0""","""8046EAFD0""","""Euro""",0
2022-09-02 00:23:00,"""Bank of Albany""","""Bank of Albany""",1.9413e10,"""8079D50F0""","""8079D50F0""","""US Dollar""",0
2022-09-02 00:23:00,"""Bank of Albany""","""Japan Bank #123""",1.4323e10,"""8079D50F0""","""81435ABC0""","""Yen""",0
2022-09-01 00:12:00,"""Germany Bank #276""","""Russia Bank #179""",1.0125e10,"""8061E29B0""","""81FD9FDB0""","""Ruble""",0
2022-09-01 00:23:00,"""France Bank #170""","""Bank of Newport""",7.7374e9,"""84319C900""","""84BD9D900""","""US Dollar""",0
2022-09-01 00:24:00,"""National Bank of Pittsburgh""","""National Bank of Pittsburgh""",7.1356e9,"""808ACD170""","""808ACD170""","""US Dollar""",0


자기가 자신한테 송금??

# 8) 피처 엔지니어링

In [ ]:
# -------------------------
# (A) 상수/룩업 리스트
# -------------------------
PAYMENT_FORMATS = ["ACH", "Cheque", "Bitcoin", "Cash", "Credit Card", "Wire", "Reinvestment"]

HIGH_RISK_SENDER_BANKS = [
    "National Bank of Dallas", "Savings Bank of Augusta", "China Bank #27",
    "India Bank #96", "Brazil Bank #128", "National Bank of Milford",
    "Savings Bank of Sacramento", "Saudi Arabia Bank #14", "Israel Bank #16",
    "Golden Credit Union"
]

HIGH_RISK_RECEIVER_BANKS = [
    "China Bank #292", "China Bank #27", "China Bank #22", "Japan Bank #143",
    "Brazil Bank #50", "Bank of Denver", "Saudi Arabia Bank #56",
    "Israel Bank #48", "The Pine Bank", "National Bank of Milford"
]

# 엔티티 타입 가중치 (v1: 단순 룰)
ENTITY_TYPE_SCORE_MAP = {
    "Corporation": 2.0,
    "Sole Proprietorship": 1.5
}

# Payment method risk (v1: 단순 룰)
# (정확한 가중치는 EDA에서 조정 가능. 지금은 “비중 높았던 수단에 가중”만 구현)
PAYMENT_RISK_MAP = {
    "ACH": 3.0,
    "Bitcoin": 2.5,
    "Cheque": 2.5,
    "Cash": 1.5,
    "Credit Card": 1.3,
    "Wire": 1.0,
    "Reinvestment": 1.0
}

In [ ]:
# -------------------------
# (B) Timestamp 파생
# -------------------------
q_feat = q_feat.with_columns([
    pl.col("ts").dt.hour().cast(pl.Int16).alias("hour"),
    pl.col("ts").dt.weekday().cast(pl.Int8).alias("day_of_week"),  # 월=1 ~ 일=7 (Polars 기본)
    pl.col("ts").dt.weekday().is_in([6, 7]).alias("is_weekend"),
    pl.col("ts").dt.date().alias("ts_day"),

    # 새벽 여부 (예: 0~5시)
    pl.col("ts").dt.hour().is_between(0, 5).alias("is_dawn"),

    # TimeofDay bucket (예시: 새벽/오전/오후/밤)
    pl.when(pl.col("ts").dt.hour().is_between(0, 5)).then(pl.lit("dawn"))
      .when(pl.col("ts").dt.hour().is_between(6, 11)).then(pl.lit("morning"))
      .when(pl.col("ts").dt.hour().is_between(12, 17)).then(pl.lit("afternoon"))
      .otherwise(pl.lit("evening"))
      .alias("timeofday_bucket"),
])

In [ ]:
# -------------------------
# (C) Amount 파생 (float32 + log)
# -------------------------
q_feat = q_feat.with_columns([
    # float32 캐스팅
    pl.col("Amount_Paid_USD").cast(pl.Float32),
    pl.col("Amount_Received_USD").cast(pl.Float32),

    # log1p
    pl.col("Amount_Paid_USD").log1p().alias("log_amount_paid_usd"),
    pl.col("Amount_Received_USD").log1p().alias("log_amount_received_usd"),

    # Round number (1000 단위)
    (pl.col("Amount_Paid_USD") % 1000 == 0).alias("is_round_1000_paid"),
    (pl.col("Amount_Received_USD") % 1000 == 0).alias("is_round_1000_received"),

    # 더 강한 라운드 (10000 단위)
    (pl.col("Amount_Paid_USD") % 10000 == 0).alias("is_round_10000_paid"),
])

In [ ]:
# -------------------------
# (D) Payment Format 원핫 + 파생
# -------------------------
# 원핫(각 format별 bool)
q_feat = q_feat.with_columns([
    (pl.col("Payment Format") == "ACH").alias("is_ach"),
    (pl.col("Payment Format") == "Cheque").alias("is_cheque"),
    (pl.col("Payment Format") == "Bitcoin").alias("is_bitcoin_fmt"),
    (pl.col("Payment Format") == "Cash").alias("is_cash"),
    (pl.col("Payment Format") == "Credit Card").alias("is_credit_card"),
    (pl.col("Payment Format") == "Wire").alias("is_wire"),
    (pl.col("Payment Format") == "Reinvestment").alias("is_reinvestment"),
])

# Crypto 여부 (format이 Bitcoin이거나 currency가 Bitcoin이면 True)
q_feat = q_feat.with_columns([
    ((pl.col("Payment Format") == "Bitcoin") | (pl.col("Payment Currency") == "Bitcoin"))
      .alias("is_crypto_transfer"),

    # High value ACH (>= 1,000,000 USD & ACH)
    ((pl.col("Payment Format") == "ACH") & (pl.col("Amount_Paid_USD") >= 1_000_000))
      .alias("is_high_value_ach"),
])

# Payment_Method_Risk (format 기반 가중치)
q_feat = q_feat.with_columns([
    pl.col("Payment Format")
      .replace(PAYMENT_RISK_MAP, default=1.0)
      .cast(pl.Float32)
      .alias("payment_method_risk"),
])

# format × amount interaction (v1용)
q_feat = q_feat.with_columns([
    (pl.col("payment_method_risk") * pl.col("log_amount_paid_usd")).alias("risk_x_log_paid"),
    (pl.col("is_ach").cast(pl.Int8) * pl.col("log_amount_paid_usd")).alias("ach_x_log_paid"),
    (pl.col("is_bitcoin_fmt").cast(pl.Int8) * pl.col("log_amount_paid_usd")).alias("btc_x_log_paid"),
])

In [ ]:
# -------------------------
# (E) Entity type 파싱 + 점수
# -------------------------
# "Corporation #26522" → "Corporation"
q_feat = q_feat.with_columns([
    pl.col("Sender_Entity")
      .cast(pl.Utf8)
      .str.replace(r"\s*#\d+.*$", "")   # "#숫자" 이후 제거
      .alias("sender_entity_type"),

    pl.col("Receiver_Entity")
      .cast(pl.Utf8)
      .str.replace(r"\s*#\d+.*$", "")
      .alias("receiver_entity_type"),
])

# 타입 점수
q_feat = q_feat.with_columns([
    pl.col("sender_entity_type")
      .replace(ENTITY_TYPE_SCORE_MAP, default=1.0)
      .cast(pl.Float32)
      .alias("sender_entity_type_score"),

    pl.col("receiver_entity_type")
      .replace(ENTITY_TYPE_SCORE_MAP, default=1.0)
      .cast(pl.Float32)
      .alias("receiver_entity_type_score"),
])


In [ ]:
# -------------------------
# (F) High risk bank flags
# -------------------------
q_feat = q_feat.with_columns([
    pl.col("Sender_Bank_Name").is_in(HIGH_RISK_SENDER_BANKS).alias("high_risk_sender_bank_flag"),
    pl.col("Receiver_Bank_Name").is_in(HIGH_RISK_RECEIVER_BANKS).alias("high_risk_receiver_bank_flag"),
])

In [ ]:
# 확인하고 싶은 핵심 피처만 뽑아서 head 5
inspect_cols = [
    # timestamp 관련
    "Timestamp", "ts", "hour", "day_of_week", "is_weekend", "is_dawn", "timeofday_bucket",

    # amount 관련
    "Amount_Paid_USD", "log_amount_paid_usd",
    "Amount_Received_USD", "log_amount_received_usd",
    "is_round_1000_paid",

    # payment format
    "Payment Format",
    "is_ach", "is_bitcoin_fmt", "is_crypto_transfer",
    "is_high_value_ach", "payment_method_risk", "risk_x_log_paid",

    # entity / bank
    "Sender_Entity", "sender_entity_type", "sender_entity_type_score",
    "Sender_Bank_Name", "high_risk_sender_bank_flag",

    # label / split
    "Is Laundering", "split"
]

q_feat.select(inspect_cols).head(5).collect()

KeyboardInterrupt: 

# 9) 모델에 안 넣을 컬럼 drop    

In [ ]:
DROP_COLS_V1 = [
    # raw time / ids
    "Timestamp", "ts", "bucket_ts", "bucket_ts_str",

    # account / bank / entity identifiers
    "From Account", "To Account",             # -> baseline-ml-v2에서는 주석 해제(계좌번호 드랍)
    # "From Bank", "To Bank",
    # "Sender_Bank_Name", "Receiver_Bank_Name",
    # "Sender_Entity", "Receiver_Entity",

    # intermediate entity cols
    "sender_entity_type", "receiver_entity_type",

    # raw amount columns
    "Amount Paid", "Amount Received",
    # "Amount_Paid_USD", "Amount_Received_USD",

    # currency
    # "Receiving Currency", "Payment Currency"

    "sender_node", "receiver_node",
]

q_model = q_feat.drop([c for c in DROP_COLS_V1 if c in q_feat.columns])

In [ ]:
# drop 전 51
print(len(q_model.columns))
print(q_model.columns)

In [ ]:
FLOAT64_COLS = [
    "log_amount_paid_usd",
    "log_amount_received_usd",
    "risk_x_log_paid",
    "ach_x_log_paid",
    "btc_x_log_paid",
]

q_model = q_model.with_columns([
    pl.col(c).cast(pl.Float32) for c in FLOAT64_COLS
])

In [ ]:
q_model.schema

In [ ]:
#“이 컬럼이 numeric / categorical인지” 빠르게 나누기
schema = q_model.schema

numeric_cols = [
    c for c, d in schema.items()
    if d in (
        pl.Int8, pl.Int16, pl.Int32, pl.Int64,
        pl.Float32, pl.Float64,
        pl.Boolean,   # ✅ 추가
    )
]

categorical_cols = [
    c for c, d in schema.items()
    if d in (pl.Utf8, pl.Categorical)
]

print("Numeric:", numeric_cols)
print("Categorical:", categorical_cols)

# 10) 마지막에 pandas로 collect

In [ ]:
split_counts = (
    q_feat.group_by("split")
          .agg(pl.count().alias("rows"))
          .collect()
)
print(split_counts)

In [ ]:
label_dist = (
    q_feat.group_by(["split", "Is Laundering"])
          .agg(pl.count().alias("rows"))
          .sort(["split", "Is Laundering"])
          .collect()
)
print(label_dist)

In [ ]:
# =========================================================
# sklearn로 넘기기 위한 "컬럼만" 선택 후 collect
# =========================================================
label_col = "Is Laundering"
META_COLS = ["ts_day"]  # KPI용 메타

feature_cols = [
    c for c in q_model.columns
    if c not in (["split", label_col] + META_COLS)
]

q_train = (
    q_model.filter(pl.col("split") == "train")
          .select(feature_cols + [label_col] + META_COLS)
)
train_df = q_train.collect().to_pandas()

q_val = (
    q_model.filter(pl.col("split") == "val")
          .select(feature_cols + [label_col] + META_COLS)
)
val_df   = q_val.collect().to_pandas()

q_test = (
    q_model.filter(pl.col("split") == "test")
          .select(feature_cols + [label_col] + META_COLS)
)
test_df  = q_test.collect().to_pandas()

print(train_df.shape, val_df.shape, test_df.shape)

# 11) Top-k 로직(하루 기준/누적 기준)

In [ ]:
# Top-k 로직 교체용 함수:“k개 적발하려면 몇 건을 봐야 하는지”를 계산하는 함수
def workload_to_find_k_positives(y_true, y_score, k_pos):
    """
    목표: true positive를 k_pos개 '찾기' 위해
         상위 점수부터 몇 건(N)을 조사해야 하는지 계산

    Returns:
      N_required: 필요한 조사 건수
      precision:  k_found / N_required
      recall:     k_found / total_pos
      f1:         f1 (top-N을 양성으로 가정한 경우)
      k_found:    실제로 찾은 TP 수 (total_pos < k_pos면 total_pos)
    """
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score)

    total_pos = y_true.sum()
    if total_pos == 0:
        return 0, 0.0, 0.0, 0.0, 0

    # 점수 내림차순 정렬
    order = np.argsort(-y_score)
    y_sorted = y_true[order]

    # 누적 TP
    cum_tp = np.cumsum(y_sorted)

    target = min(k_pos, int(total_pos))

    # cum_tp >= target 되는 첫 index
    idx = np.searchsorted(cum_tp, target, side="left")
    N_required = int(idx + 1)  # index -> count

    k_found = int(cum_tp[idx])  # 보통 target과 같음(동점/중복 없으면)

    precision = k_found / (N_required + 1e-12)
    recall = k_found / (total_pos + 1e-12)

    # top-N을 양성으로 보면: TP=k_found, FP=N-TP, FN=total_pos-TP
    fp = N_required - k_found
    fn = total_pos - k_found
    f1 = (2 * k_found) / (2 * k_found + fp + fn + 1e-12)

    return N_required, precision, recall, f1, k_found

In [ ]:
def per_day_workload_summary(df, day_col, label_col, score_col, k_pos_list=(30,50,100,200)):
    """
    df: pandas DataFrame with [day_col, label_col, score_col]
    day별로 'k_pos개 적발하기 위해 필요한 조사량 N' 계산 후
    k_pos마다 daily 분포 + mean/median/p90 요약을 반환
    """
    out_rows = []
    for day, g in df.groupby(day_col):
        y = g[label_col].astype(int).values
        s = g[score_col].values
        total_pos = int(y.sum())
        n = len(g)

        for k_pos in k_pos_list:
            N_req, p, r, f1, found = workload_to_find_k_positives(y, s, k_pos)
            out_rows.append({
                "day": day,
                "n_rows": n,
                "pos": total_pos,
                "k_pos": k_pos,
                "N_required": N_req,
                "precision_at_Nk": p,
                "recall_at_Nk": r,
                "f1_at_Nk": f1,
                "found_tp": found,
                "target_k": min(k_pos, total_pos),
                "hit_target": (found >= min(k_pos, total_pos)) and (total_pos > 0)
            })

    daily = pd.DataFrame(out_rows)

    # day에 pos=0이면 N_required=0이 되고 의미가 약하니,
    # 운영 KPI로는 "pos>0인 day"만 요약하는게 보통 더 깔끔함
    daily_pos = daily[daily["pos"] > 0].copy()

    def summarize(sub):
        # N_required는 "작을수록 좋음"
        return pd.Series({
            "days": sub["day"].nunique(),
            "mean_N_required": sub["N_required"].mean(),
            "median_N_required": sub["N_required"].median(),
            "p90_N_required": sub["N_required"].quantile(0.90),
            "mean_precision": sub["precision_at_Nk"].mean(),
            "median_precision": sub["precision_at_Nk"].median(),
            "p90_precision": sub["precision_at_Nk"].quantile(0.90),
        })

    summary = (
        daily_pos
        .groupby("k_pos", as_index=False)
        .apply(summarize)
        .reset_index(drop=True)
    )

    return daily, summary

In [ ]:
def log_per_day_kpi_to_wandb(split_name, df_raw_with_day, y_true, y_score, k_list=(30,50,100,200)):
    """
    split_name: "val" or "test"
    df_raw_with_day: pandas DF that includes column "ts_day" aligned with y_true/y_score index
    y_true/y_score: numpy-like aligned arrays
    """
    tmp = pd.DataFrame({
        "ts_day": df_raw_with_day["ts_day"].values,
        "label": np.asarray(y_true).astype(int),
        "score": np.asarray(y_score),
    })

    daily, summary = per_day_workload_summary(
        tmp,
        day_col="ts_day",
        label_col="label",
        score_col="score",
        k_pos_list=k_list
    )

    # (A) 요약 로그: k별 mean/median/p90
    for _, row in summary.iterrows():
        k = int(row["k_pos"])
        wandb.log({
            f"{split_name}_perday_find_{k}_mean_N": float(row["mean_N_required"]),
            f"{split_name}_perday_find_{k}_median_N": float(row["median_N_required"]),
            f"{split_name}_perday_find_{k}_p90_N": float(row["p90_N_required"]),
            f"{split_name}_perday_find_{k}_mean_precision": float(row["mean_precision"]),
            f"{split_name}_perday_find_{k}_median_precision": float(row["median_precision"]),
            f"{split_name}_perday_find_{k}_p90_precision": float(row["p90_precision"]),
            f"{split_name}_perday_days_with_pos_{k}": int(row["days"]),
        })

    # (B) 분포 확인용 테이블(너무 크면 상위 일부만)
    # day*k 만큼 행이 생김 -> 기간 짧으면 OK, 길면 샘플링/요약만 남기기
    table = wandb.Table(dataframe=daily.head(5000))
    wandb.log({f"{split_name}_perday_workload_table": table})

In [ ]:
def score_bin_distribution(y_true, y_score, n_bins=10, score_scale=1000):
    """
    score를 0~score_scale로 스케일링한 뒤,
    bin별 label(0/1) 건수 집계.
    항상 normal_cnt / laundering_cnt 컬럼이 존재하도록 보정.
    """
    y = np.asarray(y_true).astype(int)
    s = np.asarray(y_score)

    s_scaled = np.clip(s * score_scale, 0, score_scale)
    bins = np.linspace(0, score_scale, n_bins + 1)

    # 0..n_bins-1
    bin_idx = np.digitize(s_scaled, bins, right=False) - 1
    bin_idx = np.clip(bin_idx, 0, n_bins - 1)

    df = pd.DataFrame({"bin": bin_idx, "label": y})

    # label별 count (항상 0/1 둘 다 컬럼을 만들도록 reindex)
    cnt = (
        df.groupby(["bin", "label"])
          .size()
          .unstack(fill_value=0)
          .reindex(columns=[0, 1], fill_value=0)   # ✅ 핵심
          .rename(columns={0: "normal_cnt", 1: "laundering_cnt"})
          .reset_index()
    )

    # bin_range 붙이기
    cnt["bin_range"] = cnt["bin"].apply(lambda i: f"{int(bins[i])}-{int(bins[i+1])}")

    # 보기 좋게 정렬
    cnt = cnt[["bin", "bin_range", "normal_cnt", "laundering_cnt"]]
    return cnt

# 12-1) W&B Sweeps용 train/eval 함수 정의(xgboost)

In [ ]:
def train_eval_sweep():
    """
    W&B agent가 호출하는 sweep train function.
    - run 안에서 transformer fit은 train만
    - val/test는 transform만 (누수 방지)
    - scale_pos_weight sweep
    - val 기준으로 sweep 최적화 metric 로깅
    - PR curve / CM / Top-K는 val/test 각각 로깅
    - XGB feature importance(gain) 로깅 (변환 후 feature name)
    """
    run = wandb.init()
    config = wandb.config

    # =========================
    # (0) RAW -> X/y 분리
    # =========================
    LABEL_COL = "Is Laundering"

    # train/val/test는 "이미 time split 완료된 pandas df"라고 가정
    # (run 밖에서 만들어둔 걸 그대로 씀)
    X_train_raw = train_df.drop(columns=[LABEL_COL, "ts_day"])
    y_train     = train_df[LABEL_COL].astype(int)

    X_val_raw   = val_df.drop(columns=[LABEL_COL, "ts_day"])
    y_val       = val_df[LABEL_COL].astype(int)

    X_test_raw  = test_df.drop(columns=[LABEL_COL, "ts_day"])
    y_test      = test_df[LABEL_COL].astype(int)

    # =========================
    # (1) 컬럼 타입 분리 (run마다 동일하지만 안전하게 여기서)
    # =========================
    categorical_features = X_train_raw.select_dtypes(include=["object", "category", "string"]).columns.tolist()
    numerical_features   = [c for c in X_train_raw.columns if c not in categorical_features]

    # =========================
    # (2) Transformer 정의 & fit/transform (누수 방지 핵심)
    # =========================
    num_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        # XGBoost면 보통 스케일링 생략 가능 (원하면 config로 켜도 됨)
    ])

    cat_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("enc", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ])

    transformer = ColumnTransformer(
        transformers=[
            ("num", num_pipe, numerical_features),
            ("cat", cat_pipe, categorical_features),
        ],
        remainder="drop",
        verbose_feature_names_out=False,  # get_feature_names_out 깔끔하게
    )

    # ✅ fit은 train만
    X_train = transformer.fit_transform(X_train_raw)
    # ✅ val/test는 transform만
    X_val   = transformer.transform(X_val_raw)
    X_test  = transformer.transform(X_test_raw)

    # 변환 후 feature name 확보 (importance 로깅용)
    try:
        feature_names = transformer.get_feature_names_out()
        feature_names = [str(x) for x in feature_names]
    except Exception:
        feature_names = [f"f{i}" for i in range(X_train.shape[1])]

    # =========================
    # (A) 모델 생성 (sweep 파라미터)
    # =========================
    xgb_model = XGBClassifier(
        n_estimators=config.n_estimators,
        max_depth=config.max_depth,
        learning_rate=config.learning_rate,
        subsample=config.subsample,
        colsample_bytree=config.colsample_bytree,
        reg_lambda=config.reg_lambda,
        reg_alpha=config.reg_alpha,
        min_child_weight=config.min_child_weight,
        gamma=config.gamma,
        scale_pos_weight=config.scale_pos_weight,   # ✅ sweep
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    # =========================
    # (B) 학습
    # =========================
    xgb_model.fit(X_train, y_train)

    # =========================
    # (C) 평가 (val/test 둘 다)
    # =========================
    def safe_auc_metrics(y_true, y_score):
        if len(np.unique(y_true)) <= 1:
            return np.nan, np.nan
        return roc_auc_score(y_true, y_score), average_precision_score(y_true, y_score)

    def topk_metrics(y_true, y_score, k):
        k = min(k, len(y_true))
        idx = np.argsort(-y_score)[:k]
        y_pred = np.zeros_like(y_true)
        y_pred[idx] = 1
        p = precision_score(y_true, y_pred, zero_division=0)
        r = recall_score(y_true, y_pred, zero_division=0)
        f = f1_score(y_true, y_pred, zero_division=0)
        return p, r, f

    def threshold_at_recall(y_true, y_score, desired_recall=0.90):
        prec, rec, thr = precision_recall_curve(y_true, y_score)
        if len(thr) == 0:
            return 0.5, prec, rec, thr
        closest_i = np.argmin(np.abs(rec[1:] - desired_recall))
        return float(thr[closest_i]), prec, rec, thr

    # -------------------------
    # (C-1) val / test 확률 예측
    # -------------------------
    prob_val  = xgb_model.predict_proba(X_val)[:, 1]
    prob_test = xgb_model.predict_proba(X_test)[:, 1]

    val_roc_auc,  val_auprc  = safe_auc_metrics(y_val, prob_val)
    test_roc_auc, test_auprc = safe_auc_metrics(y_test, prob_test)

    # ✅ sweep 최적화 metric은 val 기준으로 통일
    wandb.log({
        "val_roc_auc":  val_roc_auc,
        "val_auprc":    val_auprc,
        "test_roc_auc": test_roc_auc,
        "test_auprc":   test_auprc,
        "n_train": len(y_train),
        "pos_train": int(np.sum(y_train)),
        "pos_val": int(np.sum(y_val)),
        "pos_test": int(np.sum(y_test)),
    })

    # -------------------------
    # (C-2) PR Curve 로깅 (val/test 각각)
    # -------------------------
    # PR curve 로깅 (val/test)
    prec_v, rec_v, _ = precision_recall_curve(y_val, prob_val)
    fig_pr_v = plt.figure()
    plt.plot(rec_v, prec_v)
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title(f"VAL PR Curve (AUPRC={val_auprc:.4f})")
    wandb.log({"val_pr_curve": wandb.Image(fig_pr_v)})
    plt.close(fig_pr_v)

    prec_t, rec_t, _ = precision_recall_curve(y_test, prob_test)
    fig_pr_t = plt.figure()
    plt.plot(rec_t, prec_t)
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title(f"TEST PR Curve (AUPRC={test_auprc:.4f})")
    wandb.log({"test_pr_curve": wandb.Image(fig_pr_t)})
    plt.close(fig_pr_t)

    # -------------------------
    # (C-3) Top-k
    # -------------------------
    # "K TP 찾기 위한 조사량" 로깅 (val/test 각각)
    for k_pos in [450, 750, 1500, 3000]:
        N_v, p_v, r_v, f_v, found_v = workload_to_find_k_positives(y_val.values, prob_val, k_pos)
        N_t, p_t, r_t, f_t, found_t = workload_to_find_k_positives(y_test.values, prob_test, k_pos)

        wandb.log({
            # VAL
            f"val_find_{k_pos}_N_required": N_v,
            f"val_find_{k_pos}_precision": p_v,
            f"val_find_{k_pos}_recall":    r_v,
            f"val_find_{k_pos}_f1":        f_v,
            f"val_find_{k_pos}_found_tp":  found_v,

            # TEST
            f"test_find_{k_pos}_N_required": N_t,
            f"test_find_{k_pos}_precision": p_t,
            f"test_find_{k_pos}_recall":    r_t,
            f"test_find_{k_pos}_f1":        f_t,
            f"test_find_{k_pos}_found_tp":  found_t,
        })

    # per-day KPI 로깅 (VAL/TEST 각각)
    # 전제: val_df/test_df에 ts_day 컬럼 존재
    log_per_day_kpi_to_wandb("val",  val_df,  y_val.values,  prob_val,  k_list=(30,50,100,200))
    log_per_day_kpi_to_wandb("test", test_df, y_test.values, prob_test, k_list=(30,50,100,200))

    dist_val  = score_bin_distribution(y_val.values,  prob_val,  n_bins=10, score_scale=1000)
    dist_test = score_bin_distribution(y_test.values, prob_test, n_bins=10, score_scale=1000)
    wandb.log({
        "val_score_bin_table":  wandb.Table(dataframe=dist_val),
        "test_score_bin_table": wandb.Table(dataframe=dist_test),
    })

    # -------------------------
    # (C-4) threshold@recall (val에서 threshold 선택 → test에 적용)
    # -------------------------
    # threshold@recall: val에서 threshold 선택 → test에 적용
    desired_recall = 0.90
    chosen_thr, _, _, _ = threshold_at_recall(y_val, prob_val, desired_recall)

    y_val_pred  = (prob_val  >= chosen_thr).astype(int)
    y_test_pred = (prob_test >= chosen_thr).astype(int)

    cm_val  = confusion_matrix(y_val,  y_val_pred)
    cm_test = confusion_matrix(y_test, y_test_pred)

    # confusion matrix 로깅
    fig_cm_v = plt.figure(figsize=(6,4))
    sns.heatmap(cm_val, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.title(f"VAL CM @thr={chosen_thr:.4f}")
    wandb.log({"val_confusion_matrix": wandb.Image(fig_cm_v)})
    plt.close(fig_cm_v)

    fig_cm_t = plt.figure(figsize=(6,4))
    sns.heatmap(cm_test, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.title(f"TEST CM @thr={chosen_thr:.4f}")
    wandb.log({"test_confusion_matrix": wandb.Image(fig_cm_t)})
    plt.close(fig_cm_t)

    wandb.log({"chosen_threshold": float(chosen_thr)})

    # =========================
    # (C-5) Feature Importance (gain)
    # =========================
    booster = xgb_model.get_booster()
    gain_dict = booster.get_score(importance_type="gain")

    if len(gain_dict) > 0:
        imp_rows = []
        for k, v in gain_dict.items():
            # k: "f12"
            try:
                idx = int(k[1:])
            except Exception:
                continue
            name = feature_names[idx] if idx < len(feature_names) else k
            imp_rows.append((name, float(v)))

        imp_rows.sort(key=lambda x: x[1], reverse=True)
        topN = int(getattr(config, "topn_importance", 30)) if hasattr(config, "topn_importance") else 30
        imp_top = imp_rows[:topN]

        table = wandb.Table(columns=["feature", "gain"])
        for name, gain in imp_top:
            table.add_data(name, gain)
        wandb.log({"feature_importance_gain_table": table})

        fig_imp = plt.figure(figsize=(8, 6))
        names = [x[0] for x in imp_top][::-1]
        vals  = [x[1] for x in imp_top][::-1]
        plt.barh(names, vals)
        plt.xlabel("Gain")
        plt.title(f"Top-{len(imp_top)} Feature Importance (Gain)")
        wandb.log({"feature_importance_gain_bar": wandb.Image(fig_imp)})
        plt.close(fig_imp)

    # =========================
    # (D) 콘솔 리포트 (test)
    # =========================
    print("=== Sweep Report ===")
    print("VAL  ROC-AUC:", val_roc_auc,  "VAL  AUPRC:", val_auprc)
    print("TEST ROC-AUC:", test_roc_auc, "TEST AUPRC:", test_auprc)
    print("Chosen thr(from VAL):", chosen_thr)
    print(classification_report(y_test, y_test_pred, zero_division=0))

    # =========================
    # (E) Artifacts 저장/업로드
    # # =========================
    # os.makedirs("artifacts_out", exist_ok=True)

    # preproc_path = "artifacts_out/transformer.joblib"
    # joblib.dump(transformer, preproc_path)

    # model_joblib_path = "artifacts_out/aml_model.joblib"
    # joblib.dump(xgb_model, model_joblib_path)

    # model_pkl_path = "artifacts_out/aml_model.pkl"
    # with open(model_pkl_path, "wb") as f:
    #     pickle.dump(xgb_model, f)

    # # (선택) 데이터 샘플 저장
    # data_sample_path = "artifacts_out/train_sample.csv"
    # try:
    #     sample_n = 2000
    #     X_train_raw_sample = X_train_raw.head(sample_n).copy()
    #     y_train_sample = y_train.loc[X_train_raw_sample.index].copy()
    #     X_train_raw_sample[LABEL_COL] = y_train_sample
    #     X_train_raw_sample.to_csv(data_sample_path, index=False)
    #     include_sample = True
    # except Exception:
    #     include_sample = False

    # art = wandb.Artifact(
    #     name="aml_bundle",
    #     type="model",
    #     description="Transformer(train-only fit) + XGB(scale_pos_weight sweep) + optional data sample"
    # )
    # art.add_file(preproc_path)
    # art.add_file(model_joblib_path)
    # art.add_file(model_pkl_path)
    # if include_sample:
    #     art.add_file(data_sample_path)

    # wandb.log_artifact(art)

    # run.finish()

# 12-2) Sweep Config 정의 + 실행

In [ ]:
# sweep_config = {
#     "method": "bayes",
#     "metric": {"name": "val_auprc", "goal": "maximize"},  # ★ 수정
#     "parameters": {
#         "n_estimators": {"values": [200, 400, 600]},
#         "max_depth": {"values": [3, 4, 6, 8]},
#         "learning_rate": {"values": [0.03, 0.05, 0.1]},
#         "subsample": {"values": [0.7, 0.9, 1.0]},
#         "colsample_bytree": {"values": [0.7, 0.9, 1.0]},
#         "reg_lambda": {"values": [1.0, 2.0, 5.0]},
#         "reg_alpha": {"values": [0.0, 0.1, 0.5]},
#         "min_child_weight": {"values": [1, 5, 10]},
#         "gamma": {"values": [0.0, 0.5, 1.0]},
#         "scale_pos_weight": {"values": [1, 5, 10, 20]},  # ★ 추가
#     }
# }

# # (프로젝트/엔티티 이름은 팀 기준으로 정해줘)
# sweep_id = wandb.sweep(sweep_config, project="eungyulwon")

# # count: 몇 번의 실험(run)을 돌릴지
# wandb.agent(sweep_id, function=train_eval_sweep, count=10)

# 15) (선택) Artifacts "불러오기" 예시 (필요할 때만 실행)
#     - W&B에서 best 모델 버전 지정해서 다시 로컬로 가져올 수 있음

In [ ]:
# run = wandb.init(project="aml-team-experiments", job_type="inference")
# artifact = run.use_artifact("YOUR_ENTITY/aml-team-experiments/aml_xgb_bundle:latest", type="model")
# artifact_dir = artifact.download()
# loaded_transformer = joblib.load(os.path.join(artifact_dir, "transformer.joblib"))
# loaded_model = joblib.load(os.path.join(artifact_dir, "aml_model.joblib"))
# run.finish()

# 이제 loaded_transformer/loaded_model로 동일 파이프라인 재현 가능